# FastMCP 로컬 서버 제작

일반 Python 함수를 MCP 클라이언트가 발견하고 호출할 수 있는 서버 구성 요소로 만들려면 무엇을 등록해야 할까요?

## 실행 환경 준비

Python 3.12 새 커널에서 첫 셀부터 순서대로 실행합니다. 비밀 값은 출력하지 않습니다.

## 패키지 버전 확인

설치된 FastMCP와 LangChain 버전을 출력해 아래 API가 어떤 기준에서 실행됐는지 고정합니다.

In [1]:
# 패키지 버전 확인에 필요한 값을 준비하고 결과를 확인합니다.
%pip install -q "langchain[mcp]==1.4.0" langchain-openai==1.6.0 fastmcp==4.0.3 mcp==2.2.0 python-dotenv==1.2.3
import warnings
from importlib.metadata import version
from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", message=r".*langchain\.mcp.*", category=LangChainBetaWarning)

for package in ["langchain", "langchain-openai", "fastmcp", "mcp"]:
    print(package, version(package))

Note: you may need to restart the kernel to use updated packages.
langchain 1.4.0
langchain-openai 1.6.0
fastmcp 4.0.3
mcp 2.2.0


## Before: 등록하지 않은 함수

일반 Python 함수만 정의한 뒤 `list_tools()`가 빈 목록을 반환하고 호출이 거부되는지 확인합니다.

### 코드 읽기와 비교 경계

`FastMCP('course')`는 등록 정보를 보관하는 서버 객체를 만듭니다. 일반 함수는 Python 코드 안에 존재할 뿐 클라이언트가 발견할 수 없습니다. `@mcp.tool`을 붙이면 함수 이름, docstring, 타입 힌트가 각각 도구 이름, 설명, JSON Schema로 변환되어 `list_tools()` 결과에 나타납니다. 데코레이터가 붙는 시점은 서버 실행 시점이 아니라 모듈을 읽어 함수 정의를 실행하는 시점입니다.

도구 설명은 사람용 주석만이 아니라 모델의 선택 입력입니다. ‘조회합니다’처럼 범위가 넓은 설명보다 어떤 자료를 어떤 인자로 조회하고 무엇을 반환하는지 적어야 합니다. 리소스는 읽을 URI, 프롬프트는 재사용 메시지라는 다른 발견 경로를 가지므로 실행 가능한 계산을 리소스로 위장하지 않습니다. Before에서 함수가 목록에 없고 After에서 등록된 도구가 나타나는 차이는 Python 함수의 존재와 MCP 공개 표면이 별개임을 보여 줍니다.


In [2]:
# Before: 등록하지 않은 함수에 필요한 값을 준비하고 결과를 확인합니다.
import asyncio
from fastmcp import Client, FastMCP
from fastmcp.exceptions import ToolError

before = FastMCP("등록 전 서버")

def lookup_policy(topic: str) -> str:
    return {"대출": "14일"}[topic]

async def run_before() -> tuple[list[str], str]:
    async with Client(before) as client:
        names = [item.name for item in await client.list_tools()]
        try:
            await client.call_tool("lookup_policy", {"topic": "대출"})
        except ToolError as error:
            return names, type(error).__name__

before_names, before_error = await run_before()
print("tools:", before_names)
print("error:", before_error)
assert before_names == []

tools: []
error: ToolError


## After: 도구 등록

같은 함수에 `@mcp.tool`을 적용해 목록에 등록하고, `topic` 인자를 넘겨 규정 문자열을 반환받습니다.

앞 절의 비동기 호출을 다시 사용합니다. `async def` 본문은 `await`할 때 실행되고, `async with Client(...)`는 등록 목록을 읽는 동안만 연결을 유지합니다.


In [3]:
# After: 도구 등록에 필요한 값을 준비하고 결과를 확인합니다.
after = FastMCP("등록 후 서버")

@after.tool
def lookup_policy(topic: str) -> str:
    """자료실 규정을 조회합니다. topic은 대출입니다."""
    return {"대출": "일반 도서는 14일 동안 대출할 수 있습니다."}[topic]

async def run_after() -> tuple[list[str], str]:
    async with Client(after) as client:
        names = [item.name for item in await client.list_tools()]
        result = await client.call_tool("lookup_policy", {"topic": "대출"})
        return names, result.content[0].text

after_names, after_text = await run_after()
print("tools:", after_names)
print("result:", after_text)
assert after_names == ["lookup_policy"]
assert "14일" in after_text

tools: ['lookup_policy']
result: 일반 도서는 14일 동안 대출할 수 있습니다.


## 완성 서버의 세 구성 요소 확인

완성한 서버의 도구·리소스·프롬프트 목록을 읽어 각 구성 요소가 하나씩 발견되는지 확인합니다. 이 셀은 입력 스키마를 출력하지 않습니다.

완성 서버의 목록은 도구·리소스·프롬프트가 각각 하나씩 등록됐는지 확인합니다. 서버 파일이 존재하는 것과 클라이언트가 구성 요소를 발견하는 것을 따로 검사합니다.


In [4]:
# 완성 서버의 세 구성 요소 확인에 필요한 값을 준비하고 결과를 확인합니다.
import importlib.util
from pathlib import Path

relative = Path("labs/part6/ch28-mcp/course_server.py")
server_path = next(folder / relative for folder in [Path.cwd(), *Path.cwd().parents] if (folder / relative).is_file())
assert server_path.is_file(), f"서버 파일을 찾지 못했습니다: {server_path}"
spec = importlib.util.spec_from_file_location("course_server", server_path)
course_server = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(course_server)
print("server:", course_server.mcp.name)

async def inspect_server() -> tuple[list[str], list[str], list[str]]:
    async with Client(course_server.mcp) as client:
        return (
            [item.name for item in await client.list_tools()],
            [str(item.uri) for item in await client.list_resources()],
            [item.name for item in await client.list_prompts()],
        )

print(await inspect_server())

server: 교육센터 자료실
(['lookup_policy'], ['policy://handbook'], ['explain_policy'])
